# Development Data and Label Audit

**Purpose:** audit the annotated candidate pool and frozen test identity before any domain adaptation.

This notebook does not train, fine-tune, load SinBERT, run inference, or modify either source CSV. It writes only derived audit artifacts under `outputs/development_audit/`.

In [1]:
from pathlib import Path
import hashlib
import json
import random
import pandas as pd

# Locate the repository without assuming the notebook launch directory.
repo = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'ml' / 'sentiment' / 'data' / 'processed').exists()), None)
if repo is None:
    raise FileNotFoundError('Could not locate IT22638168 repository root')
processed = repo / 'ml' / 'sentiment' / 'data' / 'processed'
audit_dir = repo / 'ml' / 'sentiment' / 'outputs' / 'development_audit'
audit_dir.mkdir(parents=True, exist_ok=True)

ground_truth_path = processed / 'PREGNANCY_ANNOTATION_GROUND_TRUTH.csv'
frozen_path = processed / 'PREGNANCY_FROZEN_TEST_SET.csv'
ground_truth = pd.read_csv(ground_truth_path)
frozen = pd.read_csv(frozen_path)

def sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

frozen_ids = set(frozen['record_id'])
development = ground_truth[~ground_truth['record_id'].isin(frozen_ids)].copy()
assert len(ground_truth) == 300
assert len(frozen) == 120
assert len(development) == 180
assert ground_truth['record_id'].is_unique
assert frozen['record_id'].is_unique
assert not development['record_id'].isin(frozen_ids).any()
assert not development['text'].duplicated(keep=False).any()
assert development['text'].notna().all() and development['text'].str.strip().ne('').all()
assert development['adjudicated_label'].isin(['CALM', 'NEUTRAL', 'DISTRESSED']).all()
print('Audit source and integrity checks passed; no source file was written.')

Audit source and integrity checks passed; no source file was written.


In [2]:
def counts(frame):
    return {
        'n': int(len(frame)),
        'by_language': {str(k): int(v) for k, v in frame['language'].value_counts().sort_index().items()},
        'by_mood': {str(k): int(v) for k, v in frame['adjudicated_label'].value_counts().sort_index().items()},
        'by_language_mood': {f'{language}_{mood}': int(n) for (language, mood), n in frame.groupby(['language', 'adjudicated_label']).size().items()},
    }

summary = {
    'ground_truth': counts(ground_truth),
    'frozen_test': counts(frozen),
    'development_candidate_pool': counts(development),
    'non_frozen_record_count': int(len(development)),
    'id_duplicate_count_ground_truth': int(ground_truth['record_id'].duplicated(keep=False).sum()),
    'exact_duplicate_text_count_development': int(development['text'].duplicated(keep=False).sum()),
    'exact_duplicate_text_count_frozen': int(frozen['text'].duplicated(keep=False).sum()),
    'frozen_development_id_overlap': int(len(frozen_ids.intersection(set(development['record_id'])))),
    'frozen_sha256': sha256(frozen_path),
    'ground_truth_sha256': sha256(ground_truth_path),
    'annotation_agreement': {
        'ground_truth_raw_percent': float((ground_truth['human_label_annotator_1'] == ground_truth['human_label_annotator_2']).mean()),
        'development_raw_percent': float((development['human_label_annotator_1'] == development['human_label_annotator_2']).mean()),
        'ground_truth_disagreements': int((ground_truth['human_label_annotator_1'] != ground_truth['human_label_annotator_2']).sum()),
        'development_disagreements': int((development['human_label_annotator_1'] != development['human_label_annotator_2']).sum()),
    },
}
(audit_dir / 'development_audit_summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(summary, indent=2, ensure_ascii=False))

{
  "ground_truth": {
    "n": 300,
    "by_language": {
      "EN": 150,
      "SI": 150
    },
    "by_mood": {
      "CALM": 57,
      "DISTRESSED": 110,
      "NEUTRAL": 133
    },
    "by_language_mood": {
      "EN_CALM": 28,
      "EN_DISTRESSED": 53,
      "EN_NEUTRAL": 69,
      "SI_CALM": 29,
      "SI_DISTRESSED": 57,
      "SI_NEUTRAL": 64
    }
  },
  "frozen_test": {
    "n": 120,
    "by_language": {
      "EN": 60,
      "SI": 60
    },
    "by_mood": {
      "CALM": 40,
      "DISTRESSED": 40,
      "NEUTRAL": 40
    },
    "by_language_mood": {
      "EN_CALM": 20,
      "EN_DISTRESSED": 20,
      "EN_NEUTRAL": 20,
      "SI_CALM": 20,
      "SI_DISTRESSED": 20,
      "SI_NEUTRAL": 20
    }
  },
  "development_candidate_pool": {
    "n": 180,
    "by_language": {
      "EN": 90,
      "SI": 90
    },
    "by_mood": {
      "CALM": 17,
      "DISTRESSED": 70,
      "NEUTRAL": 93
    },
    "by_language_mood": {
      "EN_CALM": 8,
      "EN_DISTRESSED": 33,
      "EN_N

In [3]:
# Proposed split only: no split files are created until the label-target decision is resolved.
# The proposal is stratified by language x human adjudicated mood, with 20% validation.
split_proposal = {
    'status': 'PROPOSED_ONLY_NOT_CREATED',
    'seed': 20260828,
    'source': 'non-frozen records from PREGNANCY_ANNOTATION_GROUND_TRUTH.csv',
    'stratification': ['language', 'adjudicated_label'],
    'train_fraction': 0.80,
    'validation_fraction': 0.20,
    'note': 'Human mood labels are not yet legitimate SinBERT sentiment training targets; do not train until a mapping/target decision is approved.',
    'strata': {},
}
for (language, mood), group in development.groupby(['language', 'adjudicated_label']):
    n = len(group)
    n_validation = max(1, round(n * 0.20))
    split_proposal['strata'][f'{language}_{mood}'] = {'available': int(n), 'proposed_train': int(n - n_validation), 'proposed_validation': int(n_validation)}
(audit_dir / 'development_split_proposal.json').write_text(json.dumps(split_proposal, indent=2), encoding='utf-8')
print(json.dumps(split_proposal, indent=2))

{
  "status": "PROPOSED_ONLY_NOT_CREATED",
  "seed": 20260828,
  "source": "non-frozen records from PREGNANCY_ANNOTATION_GROUND_TRUTH.csv",
  "stratification": [
    "language",
    "adjudicated_label"
  ],
  "train_fraction": 0.8,
  "validation_fraction": 0.2,
  "note": "Human mood labels are not yet legitimate SinBERT sentiment training targets; do not train until a mapping/target decision is approved.",
  "strata": {
    "EN_CALM": {
      "available": 8,
      "proposed_train": 6,
      "proposed_validation": 2
    },
    "EN_DISTRESSED": {
      "available": 33,
      "proposed_train": 26,
      "proposed_validation": 7
    },
    "EN_NEUTRAL": {
      "available": 49,
      "proposed_train": 39,
      "proposed_validation": 10
    },
    "SI_CALM": {
      "available": 9,
      "proposed_train": 7,
      "proposed_validation": 2
    },
    "SI_DISTRESSED": {
      "available": 37,
      "proposed_train": 30,
      "proposed_validation": 7
    },
    "SI_NEUTRAL": {
      "availab

## Methodological gate

The current documentation defines human `CALM / NEUTRAL / DISTRESSED` ground truth and permits the sentiment-to-mood mapping `POSITIVE → CALM`, `NEUTRAL → NEUTRAL`, `NEGATIVE → DISTRESSED` only as a diagnostic proxy. It does not define that proxy as a scientifically valid fine-tuning target.

Therefore this notebook stops before training. The smallest required decision is to approve a separate development-only target construction rule (or decide that a new mood classifier/head is required), with the decision frozen before any fine-tuning. The frozen 120-record test set remains unavailable for this decision.